# Midterm Project for COMS 4740: Analyzing Representation Bias Under Controlled Class Imbalance (CIFAR-10)

## Overview

This project investigates the impact of class imbalance on the representations learned by a convolutional neural network (CNN) trained on the CIFAR-10 dataset. We will create controlled class imbalance scenarios and analyze how the learned representations differ across classes, particularly focusing on underrepresented classes. The project will involve training a CNN on the imbalanced dataset, extracting features from the penultimate layer, and visualizing these representations using dimensionality reduction techniques like t-SNE or PCA. We will also evaluate the model's performance on the test set to understand how class imbalance affects generalization.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import matplotlib.pyplot as plt
import numpy as np
import random
import copy
import time

from torch.utils.data import DataLoader, Subset
from torch import optim
from torchvision import transforms, datasets
from scipy.special import softmax
from scipy.optimize import minimize
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split

In [ ]:
# Set up GPU

if torch.cuda.is_available():
    #args.device = torch.device("cuda")
    cudnn.benchmark = True
    device = "cuda:0"
else:
    device = "cpu"
print(f'device: {device}')

!nvidia-smi

In [ ]:
def set_random_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed )
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)

set_random_seed()

## Load CIFAR-10 Dataset

In [ ]:
# Load CIFAR-10 dataset
data_mean = (0.4914, 0.4822, 0.4465)
data_std = (0.2023, 0.1994, 0.2010)
transform_train = transforms.Compose([
    #transforms.RandomCrop(32, padding=4),
    #transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(data_mean, data_std), ])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(data_mean, data_std), ])

train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_set = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

cali_indices, test_indices = train_test_split(
    range(len(test_set)),
    test_size=0.5,
    stratify=test_set.targets, # Stratified sampling
)

cali_data = Subset(test_set, cali_indices)
test_data = Subset(test_set, test_indices)

num_classes = 10
batch_size = 128

# Dataloader
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
cali_loader = DataLoader(cali_data, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)
print(f'Training data length: {len(train_data)}, calibration data length: {len(cali_data)}, test data length: {len(test_data)}')

# Get a single batch of data
train_images, train_labels = next(iter(train_loader))
cali_images, cali_labels = next(iter(cali_loader))
test_images, test_labels = next(iter(test_loader))

print(f'Training batch shape: {train_images.shape}, calibration batch shape: {cali_images.shape}, testing batch shape: {test_images.shape}')